# N2 — Hydraulics Notebook
**Competency** C5 · **Artifact** Hydraulic Sizing Report · **Input** Family 2 CSV exports

Reproduce areas/forces/flow/power, analyze margins, verify the sizing rules.


## 1-2. Purpose & inputs
Inputs: `B4-force-area.csv`, `B5-flow-curve.csv`, `B6-pump-power.csv`. Engine constants: supply 16 MPa, relief 21 MPa, pump 36 L/min.


In [1]:
from pathlib import Path
import csv, json, math, urllib.request
# Reference CSVs live in docs/figures. In-repo this resolves directly; on Colab the
# files are pulled from raw GitHub. Students can instead drop their own demo exports in DATA.
REPO_RAW='https://raw.githubusercontent.com/alibulentkoc/parallel-kinematics-hydraulics/main/docs/figures'
DATA = Path('../figures') if Path('../figures').exists() else Path('figures')
DATA.mkdir(exist_ok=True) if DATA==Path('figures') else None
def _ensure(name):
    p=DATA/name
    if not p.exists():
        try: urllib.request.urlretrieve(f'{REPO_RAW}/{name}', p); print('fetched', name)
        except Exception as e: raise FileNotFoundError(f'{name} not found locally and fetch failed: {e}')
    return p
def load_csv(name):
    rows=[]
    with open(_ensure(name)) as f:
        reader=csv.reader(l for l in f if not l.startswith('#'))
        header=next(reader)
        for r in reader:
            if r: rows.append(dict(zip(header,r)))
    return header, rows
def col(rows,k,cast=float):
    return [cast(r[k]) for r in rows if r.get(k) not in (None,'')]


In [2]:
SUPPLY=16e6; RELIEF=21e6; PUMP_MAX_LMIN=36.0; RATED_FLOW=2.5e-4; RATED_DP=3.5e6; PI4=math.pi/4; LOAD_kN=15.0
def areas(bore,rod): ac=PI4*bore*bore; ar=ac-PI4*rod*rod; return ac,ar,(ac/ar if ar>0 else math.inf)


## 3. Reproduce
Recompute φ and forces from bore/rod and confirm against the exported sweep.


In [3]:
_,b4=load_csv('B4-force-area.csv')
row=[r for r in b4 if r['rod_mm']=='22'][0]
ac,ar,phi=areas(0.040,0.022)
print(f'reproduced phi={phi:.3f}  CSV phi={row["phi"]}')
assert abs(phi-float(row['phi']))<1e-3
Fext=SUPPLY*ac/1000; Fret=SUPPLY*ar/1000
print(f'F_ext={Fext:.1f} kN (CSV {row["Fext_kN"]})  F_ret={Fret:.1f} kN (CSV {row["Fret_kN"]})')


reproduced phi=1.434  CSV phi=1.434
F_ext=20.1 kN (CSV 20.11)  F_ret=14.0 kN (CSV 14.02)


## 4. Analyze — flow curve & pump headroom


In [4]:
_,b5=load_csv('B5-flow-curve.csv')
q_op=[float(r['Q_Lmin']) for r in b5 if r['dP_over_rated']=='0.50'][0]
print(f'flow at u=0.7, dP=0.5 rated: {q_op:.2f} L/min (engine ref 7.4)')
_,b6=load_csv('B6-pump-power.csv')
row6=[r for r in b6 if r['targetV_mps']=='0.20'][0]
reqflow=float(row6['reqFlow_Lmin']); power=float(row6['power_kW'])
print(f'required flow @0.20 m/s: {reqflow:.2f} L/min   power {power:.2f} kW')


flow at u=0.7, dP=0.5 rated: 7.42 L/min (engine ref 7.4)
required flow @0.20 m/s: 15.08 L/min   power 4.02 kW


## 5-6. Generate sizing report + verify rules
Rules: φ ≤ 1.6 · F_ext ≥ load · req flow ≤ pump max · hold pressure ≤ relief.


In [5]:
p_hold=LOAD_kN*1000/ac/1e6
checks={'phi<=1.6':phi<=1.6, 'F_ext>=load':Fext>=LOAD_kN, 'reqflow<=pumpmax':reqflow<=PUMP_MAX_LMIN, 'hold<=relief':p_hold<=RELIEF/1e6}
for k,v in checks.items(): print(f'  [{"PASS" if v else "FAIL"}] {k}')
assert all(checks.values()), 'sizing rules'
print(f'hold pressure {p_hold:.1f} MPa  <= relief {RELIEF/1e6:.0f} MPa')


  [PASS] phi<=1.6
  [PASS] F_ext>=load
  [PASS] reqflow<=pumpmax
  [PASS] hold<=relief
hold pressure 11.9 MPa  <= relief 21 MPa


## 7. Export


In [6]:
with open('sizing-report.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['metric','value','unit'])
    for k,v,unit in [('A_cap',ac*1e6,'mm2'),('A_rod',ar*1e6,'mm2'),('phi',phi,''),('F_ext',Fext,'kN'),('F_ret',Fret,'kN'),('req_flow',reqflow,'L/min'),('power',power,'kW'),('hold_pressure',p_hold,'MPa')]:
        w.writerow([k, round(v,3) if v!=math.inf else v, unit])
print('exported sizing-report.csv')


exported sizing-report.csv
